In [ ]:
# !pip install -q transformers torch tqdm scikit-learn pandas matplotlib seaborn joblib

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# RUTA_PROYECTO = '/content/drive/MyDrive/ProyectoFinal_ML'
# os.chdir(RUTA_PROYECTO)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from sklearn.metrics import classification_report, confusion_matrix, f1_score
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

In [ ]:
# Carga del Conjunto Ciego (Test Blind Set)
TEST_PATH = 'data/test_blind_set.csv'

df_test = pd.read_csv(TEST_PATH, sep=';')
df_test['texto_beto'] = df_test['texto_beto'].fillna('')

print(f"Conjunto Ciego (Hold-out) cargado.")
print(f"   Total de registros : {len(df_test):,}")
print(f"   Distribucion de clases:\n{df_test['label'].value_counts().to_string()}")

y_true = df_test['label'].values

In [ ]:
def graficar_matriz(y_real, y_pred, titulo, ax):
    cm = confusion_matrix(y_real, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Falso (0)', 'Real (1)'],
                yticklabels=['Falso (0)', 'Real (1)'])
    ax.set_title(titulo, fontsize=12, pad=10)
    ax.set_ylabel('Etiqueta Real')
    ax.set_xlabel('Predicción')

In [ ]:
# Clase Dataset personalizada para inferencia en batch
class InferDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

In [ ]:
# Carga de modelo BETO y evaluación (Perfilado de VRAM y Tiempo)
RUTA_BETO = 'data/beto_final_model' # O la ruta de checkpoint donde lo hayas exportado
resultados_beto = None

if os.path.exists(RUTA_BETO):
    print("Cargando modelo BETO...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Usando dispositivo: {device}")

    tokenizer = AutoTokenizer.from_pretrained(RUTA_BETO)
    model = AutoModelForSequenceClassification.from_pretrained(RUTA_BETO)
    model.to(device)
    model.eval()

    infer_dataset = InferDataset(df_test['texto_beto'].values, tokenizer)
    infer_loader = DataLoader(infer_dataset, batch_size=32, shuffle=False)

    print("\nIniciando inferencia (Perfilado Empírico)...")
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        
    t_start = time.time()
    
    y_pred = []
    with torch.no_grad():
        for batch in tqdm(infer_loader, desc="Evaluando"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            y_pred.extend(preds.cpu().numpy())
            
    t_end = time.time()
    t_inferencia = t_end - t_start
    
    if device.type == 'cuda':
        vram_pico = torch.cuda.max_memory_allocated() / (1024 * 1024) # en MB
    else:
        vram_pico = 0.0
        
    f1 = f1_score(y_true, y_pred, average='macro')
    
    print(f"\nResultados BETO:")
    print(f"  F1-Score Macro : {f1:.4f}")
    print(f"  Tiempo Infer.  : {t_inferencia:.4f} seg")
    print(f"  VRAM Adicional : {vram_pico:.2f} MB")
    
    resultados_beto = {
        'pred': y_pred,
        'f1': f1,
        't_inferencia': t_inferencia,
        'vram_pico': vram_pico
    }
else:
    print(f"No se encontró el modelo en: {RUTA_BETO}")

In [ ]:
# Visualización y Guardado
if resultados_beto:
    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    graficar_matriz(y_true, resultados_beto['pred'], "Matriz Confusión: BETO", ax)
    plt.tight_layout()
    plt.show()

    resumen_df = pd.DataFrame([{
        'Modelo': 'BETO',
        'F1-Score Macro': round(resultados_beto['f1'], 4),
        'Tiempo Inferencia (s)': round(resultados_beto['t_inferencia'], 4),
        'VRAM Utilizada (MB)': round(resultados_beto['vram_pico'], 2)
    }])
    
    display(resumen_df)
    resumen_df.to_csv('data/resultados_evaluacion_beto.csv', sep=';', index=False)
    print("Métricas guardadas en: data/resultados_evaluacion_beto.csv")